# Graph-based multi-hop retrieval — a sweep of experiments

`colab_run_pipeline.ipynb` runs the pipeline **once** against `config/base.yaml`. This one runs
it **N times**, each with different config values, each into its own results folder.

You declare the values to try as lists, the way a hyperparameter search does:

```python
SWEEP = {"mutual_knn_k": [4, 10, 20]}
```

`scripts/experiment.py` expands that into one experiment per value, names each one, and runs
the stages for it. Nothing is overwritten, and every folder says what produced it:

```
results/<experiment_id>/
  config.yaml      the exact config this experiment's stages read
  manifest.json    git SHA, the overrides, per-stage status and duration
  logs/            run.log plus one log per stage - logging, print(), tqdm, tracebacks
  retrieval/       *.jsonl, one per condition
  metrics/         *.csv - graph stats, summary table, by question type
```

**The expensive intermediates are shared automatically.** Embeddings and NER live in
`data/processed/p-<hash>`, the graphs in `data/graphs/g-<hash>`, where each hash covers only
the config keys that change those files. So two experiments differing only in a retrieval knob
(`pcst.topk`, say) build the graphs once — stage 2, the slow one, skips for the second. Change
`mutual_knn_k` or `metadata_graph.fields` and they each get their own, as they must.

**Before running:**
* Colab — *Runtime → Change runtime type → T4 GPU*.
* Kaggle — *Settings → Accelerator → GPU*, *Internet → On*, then *Save & Run All (Commit)* for free background execution.

Budget roughly 20–30 minutes on a T4 for the first experiment and 5–10 for each additional one
that reuses the graphs. **On Colab the clone disappears when the runtime ends** — run the last
cell to download the results, or see the final cell about keeping the caches on Drive.

## 1. Clone the repo

In [ ]:
REF = "main"  # branch for iteration, or a commit SHA to pin a sweep exactly
REPO_URL = "https://github.com/hadasy-tau/graphs_project.git"

import os

# Kaggle keeps writable state in /kaggle/working, Colab in /content, anywhere else: here.
BASE = next((d for d in ("/kaggle/working", "/content") if os.path.isdir(d)), os.getcwd())
REPO = os.path.join(BASE, "graphs_project")

if os.path.isdir(REPO):
    !cd {REPO} && git fetch --all --quiet && git checkout {REF} && git pull --ff-only || true
else:
    !git clone {REPO_URL} {REPO} && cd {REPO} && git checkout {REF}

os.chdir(REPO)  # every later cell, shell command included, runs from the repo root
!git log --oneline -1

## 2. Install dependencies

Two to three minutes. `pcst-fast` compiles from C++ source here — fine on Linux, and stage 3
cannot run without it.

In [ ]:
# requirements.txt pins the en_core_web_lg 3.7.1 wheel. That pin drags spaCy back to 3.7.x
# and numpy below 2.0 with it, a downgrade the already-running kernel only picks up after a
# restart. So install everything else from the file and let spaCy fetch the model build that
# matches whatever version it resolved - same NER model, no downgrade, no restart.
lines = [l for l in open("requirements.txt").read().splitlines() if "en-core-web" not in l]
with open("/tmp/requirements-colab.txt", "w") as f:
    f.write("\n".join(lines) + "\n")

!pip install -q -r /tmp/requirements-colab.txt
!python -m spacy download en_core_web_lg

In [ ]:
# Sanity check: fail here rather than 10 minutes into a stage.
import spacy
import torch

print("torch     :", torch.__version__)
print("GPU       :", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "NONE - Colab: Runtime > Change runtime type | Kaggle: Settings > Accelerator")

spacy.load("en_core_web_lg")
print("spaCy NER : en_core_web_lg OK")

try:
    import pcst_fast  # noqa: F401
    print("pcst_fast : OK (the real Goemans-Williamson solver)")
except ImportError:
    print("pcst_fast : MISSING - stage 3 will fail. Re-run `pip install pcst-fast` and read its error.")

In [ ]:
# The config every experiment starts from. The sweep below overrides individual keys in it;
# each experiment's merged result is saved as results/<id>/config.yaml.
print(open("config/base.yaml").read())

## 3. Shared preprocessing, once

Stage 1 — download, clean, embed, NER — depends only on `embedding.*` and `ner.*`, so every
experiment that leaves those alone reuses this one cache. Doing it here pays the ~1.3 GB model
download up front, so a mistake surfaces now rather than 20 minutes into the sweep.

This leaves a `results/00_prepare/` folder holding just the config, manifest and log — the
provenance of the shared cache. It has no `summary_table.csv`, so the comparison cells ignore it.

In [ ]:
!python -u scripts/experiment.py --id 00_prepare --stages 01

## 4. The sweep

Keys are dotted paths into `config/base.yaml`; values are the alternatives to try. They are
validated against the base config before anything runs, so a typo like `semantic.mutual_knn_k`
fails in a second instead of after a GPU hour.

`SWEEP_MODE` decides how multiple keys combine: `"grid"` runs every combination (2 × 3 = 6
experiments), `"axis"` varies one key at a time with the others left at their config default
(2 + 3 = 5). Start with `axis` when you're looking at several knobs for the first time.

The preview below is the whole matrix, with the folder name each arm will get.

In [ ]:
SWEEP = {
    # base.yaml has mutual_knn_k: null, meaning "match the entity graph's average
    # degree". These arms pin it instead, for all three graphs at once, and say
    # what that automatic choice is worth.
    "mutual_knn_k": [4, 10, 20],

    # More to try - uncomment one at a time, or several with SWEEP_MODE = "axis":
    # "entity_graph.min_shared_entities": [1, 2, 3],
    # "metadata_graph.fields": [["title", "author"], ["title"], ["title", "author", "source"]],
    # "pcst.topk": [3, 6, 9],           # retrieval-only: reuses the baseline's graphs
    # "pcst.cost_e": [0.25, 0.5],       # retrieval-only
    # "pcst.combined_cost_e": [0.25, 0.5],
}

SWEEP_MODE = "grid"       # "grid" = every combination; "axis" = one key at a time
INCLUDE_BASELINE = True   # also run an all-defaults arm, called "baseline"
STAGES = "01,02,03,04"    # add ",05" for the ablation tables: slow, it rebuilds graphs 9x
EXTRA_ARGS = ""           # e.g. "--force" to rerun every arm from scratch

import sys

sys.path.insert(0, "scripts")
from experiment import GRAPH_KEYS, apply_overrides, expand_sweep, fingerprint, load_config

arms = expand_sweep(SWEEP, mode=SWEEP_MODE, include_baseline=INCLUDE_BASELINE)
base_cfg = load_config("config/base.yaml")

print(f"{len(arms)} experiment(s):" + chr(10))
graph_ids = []
for exp_id, overrides in arms:
    # Raises on a key that is not in base.yaml, and says which one.
    merged = apply_overrides(base_cfg, overrides, allow_new_keys=False)
    graph_ids.append(fingerprint(merged, GRAPH_KEYS))
    print(f"  results/{exp_id:<26} graphs g-{graph_ids[-1][:6]}  {overrides}")

# Arms sharing a graph fingerprint build those graphs once - that is the sweep's real cost.
print()
print(f"{len(set(graph_ids))} graph build(s) for {len(arms)} experiment(s)")


## 5. Run them

One experiment after another. Output streams here live and is written to
`results/<id>/logs/` at the same time. A failing experiment is recorded in its `manifest.json`
and the sweep moves on to the next one (pass `--stop-on-error` in `EXTRA_ARGS` to change that).

Re-running this cell is cheap: each stage 3 script skips a condition it has already written, and
stage 2 reuses graphs it has already built — so a Colab disconnect costs you only the stage that
was in flight.

In [ ]:
import json
import shlex

args = (f"--sweep-json {shlex.quote(json.dumps(SWEEP))} --sweep-mode {SWEEP_MODE} "
        f"{'--include-baseline ' if INCLUDE_BASELINE else ''}--stages {STAGES} {EXTRA_ARGS}")

!python -u scripts/experiment.py {args}
print("\nexit code:", _exit_code, "(0 = every experiment finished every stage)")

In [ ]:
# Hand-picked combinations instead of a product - use this when you want specific
# pairings rather than every one. Ids are optional; leave them out and they are
# derived from the overrides as above.
#
# EXPERIMENTS = [
#     {"id": "knn10_topk9",  "overrides": {"mutual_knn_k": 10, "pcst.topk": 9}},
#     {"id": "title_only",   "overrides": {"metadata_graph.fields": ["title"]}},
# ]
# spec = shlex.quote(json.dumps(EXPERIMENTS))
# !python -u scripts/experiment.py --experiments-json {spec} --stages {STAGES}


## 6. Compare the experiments

Every `results/*/metrics/summary_table.csv`, stacked with the experiment id and its overrides,
then pivoted so each condition is a row and each experiment a column. Also saved to
`results/comparison_all_experiments.csv`.

In [ ]:
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)

frames = []
for summary in sorted(Path("results").glob("*/metrics/summary_table.csv")):
    exp_dir = summary.parents[1]
    manifest_path = exp_dir / "manifest.json"
    manifest = json.loads(manifest_path.read_text(encoding="utf-8")) if manifest_path.exists() else {}
    df = pd.read_csv(summary)
    df.insert(0, "experiment_id", exp_dir.name)
    df.insert(1, "overrides", json.dumps(manifest.get("overrides", {}), sort_keys=True))
    df.insert(2, "run_status", manifest.get("status", "unknown"))
    frames.append(df)

comparison = pd.concat(frames, ignore_index=True)
comparison.to_csv("results/comparison_all_experiments.csv", index=False)

print("=== every condition, every experiment ===")
display(comparison.sort_values(["condition", "experiment_id"]))

print("=== F1 ===")
display(comparison.pivot(index="condition", columns="experiment_id", values="f1"))

print("=== evidence recall ===")
display(comparison.pivot(index="condition", columns="experiment_id", values="evidence_recall"))

In [ ]:
# Did the knob actually change the graph? For a mutual_knn_k sweep the semantic and
# metadata edge counts are the direct read-out; for metadata_graph.fields, the metadata
# row alone. If these are identical across arms the override never reached the builders,
# and every metric above is measuring the same graph.
stats = []
for path in sorted(Path("results").glob("*/metrics/graph_stats.csv")):
    df = pd.read_csv(path)
    df.insert(0, "experiment_id", path.parents[1].name)
    stats.append(df)

graph_stats = pd.concat(stats, ignore_index=True)
display(graph_stats.pivot(index="name", columns="experiment_id", values="n_edges"))
display(graph_stats.pivot(index="name", columns="experiment_id", values="oracle_connectivity"))


## 7. One experiment in detail

Its config, what each stage cost, and every metrics table it produced.

In [ ]:
EXPERIMENT_ID = arms[0][0]  # or a folder name, e.g. "mutual_knn_k10"

exp_dir = Path("results") / EXPERIMENT_ID
manifest = json.loads((exp_dir / "manifest.json").read_text(encoding="utf-8"))

print(f"=== {EXPERIMENT_ID}: {manifest['status']} ===")
print(f"overrides {json.dumps(manifest['overrides'], sort_keys=True)}")
print(f"git       {manifest['git']['sha'][:10]} on {manifest['git']['branch']}"
      f"{' (dirty)' if manifest['git']['dirty'] else ''}")
print(f"graphs    {manifest['paths']['graphs']}\n")
display(pd.DataFrame(manifest["stage_runs"])[["script", "status", "duration_s"]])

print(f"\n=== {exp_dir / 'config.yaml'} ===")
print((exp_dir / "config.yaml").read_text(encoding="utf-8"))

for csv in sorted((exp_dir / "metrics").glob("*.csv")):
    print(f"\n=== {csv} ===")
    display(pd.read_csv(csv))

print("\nlogs:", ", ".join(p.name for p in sorted((exp_dir / "logs").glob("*.log"))))

## 8. Download everything

All of `results/` — every experiment's metrics, retrieval output, logs and the config that
produced them. The caches under `data/` are left behind: hundreds of MB, and regenerated from
the model checkpoints.

In [ ]:
import shutil

BUNDLE = os.path.join(BASE, "graphs_project_experiments")
shutil.rmtree(BUNDLE, ignore_errors=True)
shutil.copytree("results", BUNDLE)

zip_path = shutil.make_archive(BUNDLE, "zip", BUNDLE)
print(f"{zip_path}  ({os.path.getsize(zip_path) / 1e6:.1f} MB)")

try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("Kaggle: download it from the Output tab.")

## Keeping the caches between sessions

`data/` dies with the Colab runtime, so the next session re-downloads and re-embeds everything.
To keep it, mount Drive and point the caches at it **before** section 3 — the fingerprinted
folder names mean nothing goes stale:

```python
from google.colab import drive
drive.mount("/content/drive")
os.environ["GRAPHS_PROJECT_DATA_ROOT"] = "/content/drive/MyDrive/graphs_project/data"
```

`results/` stays inside the clone; the download cell above is how you keep it. Both are
gitignored, so the only committed record of a sweep is the `SWEEP` cell — keep it committed, and
pin `REF` to a commit SHA if you want a manifest's git SHA to still mean something later.